### Spiel, das Gitter zur Darstellung nutzt

Die Klasse Game hat Methoden`place(pos, size=None, color=None)` und `move(old, new)`
zum Setzen und Verschieben von Steinen.
Dabei kann jeweils eine Farbe und Gr&ouml;ssen gew&auml;hlt werden, erlaubte Werte sind
`None`, `1`, `2`, oder  `3`.


Wir geben der Klasse View ein GridHelper-Objekt.  
```python
self.gridhelper = GridHelper(*grid_spec)
```

Die Klasse GridHelper hat folgende Methoden zum Zeichnen auf eine Leinwand und
zum Umrechnen von Canvaskoordinaten in Gitterpositionen.
```python
def is_inside(self, pos):
    '''testet, ob Gitterfeld pos=(col, row) innerhalb des Gitters'''
    col, row = pos
    return 0 <= col < self.ncol and 0 <= row < self.nrow

def xy2cr(self, x, y, strict=False):
    '''liefert Gitterfeld (col, row) in dem (x, y) liegt.
       Falls strict=True, wird None geliefert, falls (x,y) nicht im Gitter
    '''

def cr2xy(self, col, row,  center=False):
    '''liefert (x, y) der linken oberen Ecke des Gitterfeldes (col, row),
       oder der Feldmitte, falls center=True
    '''
   
def draw_grid(self, canvas, line_width=None, color=None):
    '''zeichnet Gitter mit geg. grid_spec'''
  
def fill_circle(self, canvas, pos, radius=2/3, color=None):
    '''zeichnet Kreisscheibe ins Gitterfeld pos=(col, row) mit
       Radius: radius*self.r_incircle
    '''
    
def stroke_circle(self, canvas, pos, radius=2/3, line_width=None, color=None):
    '''zeichnet Kreisscheibe ins Gitterfeld pos=(col, row) mit
       Radius: radius*self.r_incircle
    '''
    
def fill_rect(self, canvas, pos, color=None):
    '''fuellt das Gitterfeld pos=(col, row) mit der Farbe color'''
    
def stroke_rect(self, canvas, pos, line_width=None, color=None):
    '''fuellt das Gitterfeld pos=(col, row) mit der Farbe color'''
   
def clear_rect(self, canvas, pos):
    '''loecht das Gitterfeld pos=(col, row)'''
    
def fill_polygon(self, canvas, pos, pts, color=None):
    '''zeichnet ein Polygon ins Gitterfeld pos=(col, row),
       pts ist die Liste der Polygonpunkte, skaliert fuer ein 1x1 Feld
       (verbunden werden die Punkte [(x0+x*dx, y0+y*dy) for x, y in pts], wo (x0, y0) = cr2xy(*pos, grid_spec))
    '''
    
def stroke_polygon(self, canvas, pos, pts, color=None):
    '''zeichnet ein Polygon ins Gitterfeld pos=(col, row),
       pts ist die Liste der Polygonpunkte, skaliert fuer ein 1x1 Feld
       (verbunden werden die Punkte [(x0+x*dx, y0+y*dy) for x, y in pts], wo (x0, y0) = cr2xy(*pos, grid_spec))
    '''
    
def fill_text(self, canvas, text, pos, color=None, margin=0.1):
    '''platziert den Text text im Gitterfeld pos=(col, row)'''
```    

In [ ]:
from model_view_controller import Observable, notify, BaseView, Controller
from gridhelper import GridHelper


class Game(Observable):
    def __init__(self):
        self.ncol = 4
        self.nrow = 2
        self.options = (None, 1, 2, 3)  # options for color and size
        self.stones = {}

    @notify
    def new_game(self):
        self.stones.clear()

    def is_inside(self, pos):
        return 0 <= pos[0] < self.ncol and 0 <= pos[1] < self.nrow

    def can_place(self, pos, size=None, color=None):
        return (self.is_inside(pos) and pos not in self.stones
                and all(val in self.options for val in (size, color))
                )

    def can_move(self, old, new):
        return (self.is_inside(old) and self.is_inside(new)
                and old in self.stones and new not in self.stones
                )

    @notify
    def move(self, old, new):
        if self.can_move(old, new):
            self.stones[new] = self.stones.pop(old)
            return self.stones

    @notify
    def place(self, pos, size=None, color=None):
        if self.can_place(pos, size, color):
            self.stones[pos] = (size or 1, color or 1)
            return pos, self.stones[pos]



class View(BaseView):
    def __init__(self, game, width=100, height=100, nlayers=3, debug=True):
        super().__init__(game, width, height, nlayers, debug)
        self.bg, self.fg, self.top = self.mcanvas

        self.colors = {1: 'red', 2: 'green', 3: 'blue'}
        self.sizes = {1: 0.25, 2: 0.5, 3: 0.9}

        # grid_spec erstellen
        margin = 20
        dx = (width - 2*margin) / game.ncol
        dy = (height - 2*margin) / game.nrow
        grid_spec = (margin, margin, dx, dy, game.ncol, game.nrow)

        # GridHelper-Instanz erstellen
        self.gridhelper = GridHelper(*grid_spec)
        self.gridhelper.draw_grid(self.fg, line_width=2, color='blue')

        self.log('Drawing the Grid')

    def show_msg(self, msg):
        '''schreibe msg auf Top-Level der Canvas'''
        self.top.clear()
        self.top.fill_text(msg, 20, 15)

    def draw_stones(self, stones):
        self.bg.clear()
        for pos, (size, color) in stones.items():
            self.gridhelper.fill_circle(self.bg, pos, self.sizes[size], color=self.colors[color])

    def update(self, event, data):
        self.log(f'running update(event={event}, data={data})')
        if event == 'new_game':
            self.bg.clear()
            self.top.clear()

        if event == 'move' and data:
            self.draw_stones(data)

        if event == 'place' and data:
            pos, (size, color) = data
            self.gridhelper.fill_circle(self.bg, pos, self.sizes[size], color=self.colors[color])

In [ ]:
game = Game()
view = View(game)
view

In [ ]:
game.place((1, 1), 3, 2)

In [ ]:
game.move((1, 1), (2, 1))

In [ ]:
game.new_game()

In [ ]:
view.show_msg('Test')

### Controller
Der Controller erlaubt 3 Varianten von Callbacks.
1. An Tasten gebundene Callbacks ohne Argumente.
   ```python
   {'n': game.new_game}
   ```
2. Callbacks f&uuml;r die Mausevents `on_mouse_down`,  `on_mouse_up`, `on_mouse_out` und `on_mouse_move`.
   Der Key im Callback-Dict ist das Mausevent ohne `on_`.
   Der Callback hat die Signatur
   ```python
   def on_mouse_down(self, x, y, state):
       ...
   ```
   `self` wird an den Controller gebunden, `(x, y)` ist die Mausposition und `state` ein
   Dict, der Kommunikation zw. Callbacks erlaubt.
3. Ein Callback `key_handler` mit der Signatur
   ```python
   def key_handler(self, key, state):
       ...
   ```
   Dieses Callback muss dem Argument `key_handler` des Controllers zugewiesen werden:
   ```python
   Controller(game, view, callbacks=callbacks, key_handler=key_handler)
   ```

   

In [ ]:
def key_handler(self, key, state):
    '''Druecken von Escape wechselt zw. Colormode und Sizemode.
       Im entsprechenden Mode kann mit 1,2,3 color, bez. size gewaehlt werden.
    '''
    if key == 'Escape':
        state['colormode'] = not state.get('colormode', False)
        msg = 'set color' if state['colormode'] else 'set size'
        self.view.show_msg(msg)

    if key in '123' and 'colormode' in state:
        if state['colormode']:
            state['color'] = int(key)
        else:
            state['size'] = int(key)


game = Game()


def on_mouse_down(self, x, y, state):
    '''versucht einen Stein an der geklickten Gitterposition zu setzen
       color und size werden falls vorhanden aus dem Dict state genommen
    '''
    pos = self.view.gridhelper.xy2cr(x, y)  # in col, row umrechnen
    self.game.place(pos, state.get('size', None), state.get('color', None))


def on_mouse_down1(self, x, y, state):
    '''Klicken auf einen Stein speichert seine Position als Wert zu state['grabbed']
       Klicke auf ein leeres Feld plaziert einen Stein
    '''
    pos = self.view.gridhelper.xy2cr(x, y)
    if pos in self.game.stones:
        state['grabbed'] = pos
    else:
        self.game.place(pos, state.get('size', None), state.get('color', None))


def on_mouse_up(self, x, y, state):
    '''entfernt falls vorhanden den Key 'grabbed' aus state
       und versucht, diesen Stein an die Loslassposition zu verschieben
    '''
    pos = self.view.gridhelper.xy2cr(x, y)
    old_pos = state.pop('grabbed', None)

    if old_pos is not None:
        self.game.move(old_pos, pos)


callbacks = {
    'n': game.new_game,
    'mouse_down': on_mouse_down,
    # 'mouse_down': on_mouse_down1,
    # 'mouse_up': on_mouse_up,
    }


view = View(game)
controller = Controller(game, view, callbacks=callbacks, key_handler=key_handler)
controller